# Attendance and Absenteeism Analysis

## Project Overview
Analyze attendance records to identify absenteeism patterns by team, season, shift, and employee segment.

## Learning Objectives
- Calculate absenteeism rates across organizational and temporal dimensions
- Identify seasonal and day-of-week patterns in absences
- Compare absenteeism by shift type, team, and tenure
- Quantify the operational impact of absenteeism

## Problem Statement
Operations management needs to understand absenteeism patterns to improve staffing plans, reduce productivity loss, and identify teams or periods with chronic attendance issues.

## Why This Project Matters
Unplanned absences disrupt operations, increase overtime costs, and burden remaining staff. Understanding patterns enables proactive scheduling and targeted interventions.

## Dataset Overview
Synthetic attendance dataset: ~10,000 absence records across 500 employees over 12 months with reason codes, shift, and team data.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_employees = 500
n_records = 10000

emp_ids = np.random.choice([f'E{i:04d}' for i in range(n_employees)], n_records)
teams = {f'E{i:04d}': np.random.choice(['Engineering', 'Sales', 'Operations', 'Support', 'Marketing', 'Finance'])
         for i in range(n_employees)}
shifts = {f'E{i:04d}': np.random.choice(['Day', 'Evening', 'Night'], p=[0.55, 0.30, 0.15])
          for i in range(n_employees)}
tenure_map = {f'E{i:04d}': np.clip(np.random.exponential(3), 0.5, 15).round(1)
              for i in range(n_employees)}

dates = pd.date_range('2024-01-01', '2024-12-31', freq='D')
abs_dates = np.random.choice(dates, n_records)

reasons = np.random.choice(['Sick Leave', 'Personal', 'Family Emergency', 'No-Call No-Show',
                              'Medical Appointment', 'Mental Health Day', 'Other'],
                             n_records, p=[0.30, 0.20, 0.12, 0.08, 0.12, 0.10, 0.08])

df = pd.DataFrame({
    'RecordID': range(n_records),
    'EmployeeID': emp_ids,
    'AbsenceDate': abs_dates,
    'Team': [teams[e] for e in emp_ids],
    'Shift': [shifts[e] for e in emp_ids],
    'TenureYears': [tenure_map[e] for e in emp_ids],
    'Reason': reasons,
    'HoursAbsent': np.clip(np.random.lognormal(1.5, 0.5, n_records), 1, 12).round(1),
})
df['DayOfWeek'] = df['AbsenceDate'].dt.day_name()
df['Month'] = df['AbsenceDate'].dt.month
df['Quarter'] = df['AbsenceDate'].dt.quarter
df['TenureBand'] = pd.cut(df['TenureYears'], bins=[0, 1, 3, 6, 99], labels=['<1yr', '1-3yr', '3-6yr', '6yr+'])

print(f'Dataset: {df.shape}')
print(f'Unique employees: {df["EmployeeID"].nunique()}')
print(f'Date range: {df["AbsenceDate"].min()} to {df["AbsenceDate"].max()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nReason distribution:\n{df["Reason"].value_counts()}')
print(f'\nTeam distribution:\n{df["Team"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('Team')['RecordID'].count().sort_values().plot.barh(ax=axes[0,0], edgecolor='black', color='coral')
axes[0,0].set_title('Absence Count by Team')

dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df.groupby('DayOfWeek')['RecordID'].count().reindex(dow_order).plot.bar(ax=axes[0,1], edgecolor='black', color='steelblue')
axes[0,1].set_title('Absences by Day of Week')
axes[0,1].tick_params(axis='x', rotation=45)

monthly = df.groupby('Month')['RecordID'].count()
monthly.plot(ax=axes[1,0], marker='o', color='teal')
axes[1,0].set_title('Monthly Absence Trend')
axes[1,0].set_xlabel('Month')

df.groupby('Reason')['HoursAbsent'].mean().sort_values().plot.barh(ax=axes[1,1], edgecolor='black', color='goldenrod')
axes[1,1].set_title('Avg Hours Absent by Reason')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Absenteeism Rate by Team

In [ ]:
team_stats = df.groupby('Team').agg(
    absence_count=('RecordID', 'count'),
    unique_employees=('EmployeeID', 'nunique'),
    total_hours=('HoursAbsent', 'sum'),
    avg_hours_per_absence=('HoursAbsent', 'mean'),
).round(1)
team_stats['absences_per_employee'] = (team_stats['absence_count'] / team_stats['unique_employees']).round(1)
print('Absenteeism by Team:')
print(team_stats.sort_values('absences_per_employee', ascending=False))

## Shift Analysis

In [ ]:
shift_stats = df.groupby('Shift').agg(
    absences=('RecordID', 'count'),
    employees=('EmployeeID', 'nunique'),
    avg_hours=('HoursAbsent', 'mean'),
).round(1)
shift_stats['absences_per_employee'] = (shift_stats['absences'] / shift_stats['employees']).round(1)
print('Absenteeism by Shift:')
print(shift_stats)

fig, ax = plt.subplots(figsize=(8, 5))
shift_stats['absences_per_employee'].plot.bar(ax=ax, edgecolor='black', color='steelblue')
ax.set_title('Absences per Employee by Shift')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'shift_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

## Seasonal Patterns

In [ ]:
quarter_reason = df.groupby(['Quarter', 'Reason'])['RecordID'].count().unstack(fill_value=0)
print('Absences by Quarter × Reason:')
print(quarter_reason)

fig, ax = plt.subplots(figsize=(10, 6))
quarter_reason.plot.bar(ax=ax, edgecolor='black', stacked=True)
ax.set_title('Absence Reasons by Quarter')
ax.set_xlabel('Quarter')
ax.tick_params(axis='x', rotation=0)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'seasonal_patterns.png'), dpi=100, bbox_inches='tight')
plt.show()

## Chronic Absenteeism — Top Employees

In [ ]:
emp_abs = df.groupby('EmployeeID').agg(
    total_absences=('RecordID', 'count'),
    total_hours=('HoursAbsent', 'sum'),
    top_reason=('Reason', lambda x: x.mode().iloc[0]),
    team=('Team', 'first'),
).sort_values('total_absences', ascending=False)

print(f'Avg absences per employee: {emp_abs["total_absences"].mean():.1f}')
print(f'Std: {emp_abs["total_absences"].std():.1f}')
chronic_threshold = emp_abs['total_absences'].mean() + 2 * emp_abs['total_absences'].std()
print(f'Chronic threshold (mean + 2σ): {chronic_threshold:.0f}')
chronic = emp_abs[emp_abs['total_absences'] > chronic_threshold]
print(f'Chronic absentees: {len(chronic)}')
print(chronic.head(10))

## Tenure Impact

In [ ]:
tenure_abs = df.groupby('TenureBand').agg(
    absences=('RecordID', 'count'),
    employees=('EmployeeID', 'nunique'),
    avg_hours=('HoursAbsent', 'mean'),
).round(1)
tenure_abs['per_employee'] = (tenure_abs['absences'] / tenure_abs['employees']).round(1)
print('Absenteeism by Tenure:')
print(tenure_abs)

## Interpretation and Business Insight
- **Monday and Friday** show elevated absences — classic "long weekend" pattern worth monitoring
- **Sick leave** dominates absence reasons, but **No-Call No-Show** events signal deeper engagement issues
- **Night shift** workers tend to have higher absence rates — schedule fatigue and work-life balance are factors
- **Seasonal spikes** in Q1 (winter illness) and Q4 (holiday period) are expected but should inform staffing buffers
- **Chronic absentees** (outliers) represent a small group with outsized impact — targeted follow-up is warranted
- **New employees** (<1yr) may show different patterns due to probation policies

## Limitations
- Synthetic data with uniform random dates
- No distinction between planned and unplanned absences
- No cost data (overtime, temp staffing)
- No manager or policy context
- No return-to-work or pattern investigation outcomes

## How to Improve This Project
- Add cost modeling for each absence type
- Include planned vs unplanned classification
- Track Bradford Factor scores
- Add manager-level absence dashboards
- Include weather/flu season correlation analysis

## Production Considerations
- Weekly absence dashboards for operations managers
- Automated Bradford Factor alerts for chronic absentees
- Seasonal staffing buffer recommendations
- Integration with HRIS for real-time tracking

## Common Mistakes
- Treating all absences equally (planned PTO ≠ no-call no-show)
- Not normalizing by headcount when comparing teams
- Ignoring seasonal patterns in absence rate targets
- Punitive approaches without understanding root causes

## Mini Challenge / Exercises
1. Calculate the Bradford Factor for the top 10 most-absent employees.
2. Which Team × Shift combination has the highest absence rate?
3. Estimate the cost of absenteeism assuming $35/hour average and 1.5x overtime for coverage.
4. Identify employees whose absence pattern clusters on Mondays/Fridays.

## Final Summary / Key Takeaways
- Absenteeism analysis reveals actionable patterns in when, where, and who is absent
- Day-of-week and seasonal patterns inform proactive staffing
- Chronic absenteeism affects a small group but has outsized operational impact
- Shift and tenure segmentation provides targeted intervention opportunities
- Data-driven absence management reduces costs and improves fairness